In [3]:
import sys
import os
import json
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch

In [ ]:
from importlib import reload

sys.path.append(os.path.abspath("../"))

import Environment.env_utils
import Evaluation.evaluation_util
import Evaluation.pipeline_evaluation_shared

reload(Environment.env_utils)
reload(Evaluation.evaluation_util)
reload(Evaluation.pipeline_evaluation_shared)

loaded environment variables from: /local/scratch/jhehli/ForkSight/Environment/.env


<module 'Evaluation.pipeline_evaluation_shared' from '/local/scratch/jhehli/ForkSight/Evaluation/pipeline_evaluation_shared.py'>

In [ ]:
Environment.env_utils.load_forksight_env()

RAW_DATA_DIR = os.getenv("RAW_DATA_DIR")
HIGHRES_IMG_DIR_NAME = os.getenv("HIGHRES_IMG_DIR_NAME", "images_4096")
HIGHRES_MASK_DIR_NAME = os.getenv("HIGHRES_MASK_DIR_NAME", "masks_4096")

if RAW_DATA_DIR is None:
    raise ValueError("RAW_DATA_DIR environment variable not set.")
RAW_DATA_DIR = Path(RAW_DATA_DIR)

NNUNET_RAW_DIR = os.getenv("NNUNET_RAW_DIR")
NNUNET_RESULTS_DIR = os.getenv("NNUNET_RESULTS_DIR")
NNUNET_RAW_DIR = Path(NNUNET_RAW_DIR)
NNUNET_RESULTS_DIR = Path(NNUNET_RESULTS_DIR)

NNUNET_TEST_IMG_DIR = os.getenv("NNUNET_TEST_IMG_DIR")
NNUNET_TEST_MASK_DIR = os.getenv("NNUNET_TEST_MASK_DIR")

NNUNET_DATASET_NAME = "Dataset001_Segmentation_v1"
NNUNET_TRAINER_DIR_NAME = "nnUNetTrainerWandb__nnUNetPlans__2d"

IMAGES_DIR = NNUNET_RAW_DIR / NNUNET_DATASET_NAME / NNUNET_TEST_IMG_DIR
GROUNDTRUTH_DIR = NNUNET_RAW_DIR / NNUNET_DATASET_NAME / NNUNET_TEST_MASK_DIR
MAPPING_JSON = NNUNET_RAW_DIR / NNUNET_DATASET_NAME / "case_mapping.json"
PRED_DIR = NNUNET_RESULTS_DIR / NNUNET_DATASET_NAME / \
    NNUNET_TRAINER_DIR_NAME / "best_configuration_inference_output"

SPLIT = "test"

In [ ]:
with open(MAPPING_JSON) as f:
    mapping = json.load(f)

mapping = [e for e in mapping if e.get("split") == SPLIT]
print(f"Loaded {len(mapping)} entries" +
      (f" for split '{SPLIT}'" if SPLIT else ""))

# Group entries by original_filename
groups: dict[str, list] = defaultdict(list)
for entry in mapping:
    groups[entry["original_filename"]].append(entry)

# Separate full images (patched from a large original) and SOI patches
full_image_groups = {name: entries for name,
                     entries in groups.items() if "_soi" not in name}
soi_groups = {name: entries for name,
              entries in groups.items() if "_soi" in name}

print(f"Full images : {len(full_image_groups)}")
print(f"SOI images  : {len(soi_groups)}")

In [ ]:
def get_nnunet_case_files(entry: dict) -> tuple:
    case = entry["nnunet_case"]
    pred_path = PRED_DIR / f"{case}.png"
    pred_probs_path = PRED_DIR / f"{case}.npz"

    if not pred_path.exists() or not pred_probs_path.exists():
        raise FileNotFoundError(f"Missing files for case '{case}'.")

    return pred_path, pred_probs_path

## Evaluate full (patched) images

In [ ]:
PATCH_SIZE = (1024, 1024)
GRID_SIZE = (4, 4)

hard_dice_scores_smallobjremoved, hard_dice_scores_inclsmallobj = [], []
iou_scores_smallobjremoved, iou_scores_inclsmallobj = [], []
clDice_scores_smallobjremoved, clDice_scores_inclsmallobj = [], []
tprecs_smallobjremoved, tprecs_inclsmallobj = [], []
tsenss_smallobjremoved, tsenss_inclsmallobj = [], []

for original_filename, entries in sorted(full_image_groups.items()):
    if original_filename != "20250409_veronica_apple_tileset_14_tile_014_006.png":
        continue  # DEBUG: only evaluate one case for now

    print(f"\n" + "=" * 60 + f"\nEvaluating: {original_filename}")

    entries_sorted = sorted(entries, key=lambda e: int(e["patch"]))

    pred_patches, pred_probs_patches = [], []
    skip = False
    for entry in entries_sorted:
        pred_path, pred_probs_path = get_nnunet_case_files(entry)
        pred_patches.append(
            Evaluation.pipeline_evaluation_shared.load_binary_mask_as_tensor(pred_path))
        npz_data = np.load(pred_probs_path)
        key = "probabilities" if "probabilities" in npz_data else npz_data.files[0]
        probs_np = npz_data[key]  # (num_classes, 1, H, W)
        fg_np = probs_np[1] if probs_np.shape[0] > 1 else probs_np[0]
        pred_probs_patches.append(torch.from_numpy(
            fg_np.astype(np.float32)))  # (1, H, W)
    if skip:
        continue

    # Load full-resolution GT and original image directly from RAW_DATA_DIR
    full_img_path = RAW_DATA_DIR / HIGHRES_IMG_DIR_NAME / original_filename
    full_gt_path = RAW_DATA_DIR / HIGHRES_MASK_DIR_NAME / original_filename
    original_img = Evaluation.evaluation_util.load_transform_image(
        full_img_path, is_full_image=True).detach().cpu()
    gt_stitched = Evaluation.evaluation_util.load_transform_image(
        full_gt_path, is_mask=True, is_full_image=True).detach().cpu()

    pred_masks = torch.stack(pred_patches, dim=0)           # (B, 1, H, W)
    pred_probs = torch.stack(pred_probs_patches, dim=0)     # (B, 1, H, W)

    metrics_cleaned, metrics_raw = Evaluation.pipeline_evaluation_shared.evaluate_full_image_patches(
        pred_masks, gt_stitched, original_img,
        patch_size=PATCH_SIZE,
        grid_size=GRID_SIZE,
        output_probs=pred_probs,
    )

    hard_dice_scores_smallobjremoved.append(metrics_cleaned[0])
    hard_dice_scores_inclsmallobj.append(metrics_raw[0])
    iou_scores_smallobjremoved.append(metrics_cleaned[1])
    iou_scores_inclsmallobj.append(metrics_raw[1])
    clDice_scores_smallobjremoved.append(metrics_cleaned[2])
    clDice_scores_inclsmallobj.append(metrics_raw[2])
    tprecs_smallobjremoved.append(metrics_cleaned[3])
    tprecs_inclsmallobj.append(metrics_raw[3])
    tsenss_smallobjremoved.append(metrics_cleaned[4])
    tsenss_inclsmallobj.append(metrics_raw[4])

if hard_dice_scores_smallobjremoved:
    print("\nFull Image Evaluation Results (with vs. without small object removal)")
    print("=" * 50)
    print(
        f"Mean Dice  (removed): {Evaluation.evaluation_util.format_score(np.mean(hard_dice_scores_smallobjremoved))}")
    print(
        f"Mean Dice  (kept):    {Evaluation.evaluation_util.format_score(np.mean(hard_dice_scores_inclsmallobj))}")
    print(
        f"Mean IoU   (removed): {Evaluation.evaluation_util.format_score(np.mean(iou_scores_smallobjremoved))}")
    print(
        f"Mean IoU   (kept):    {Evaluation.evaluation_util.format_score(np.mean(iou_scores_inclsmallobj))}")
    print(
        f"Mean clDice(removed): {Evaluation.evaluation_util.format_score(np.mean(clDice_scores_smallobjremoved))}")
    print(
        f"Mean clDice(kept):    {Evaluation.evaluation_util.format_score(np.mean(clDice_scores_inclsmallobj))}")
    print(
        f"Mean tprec (removed): {Evaluation.evaluation_util.format_score(np.mean(tprecs_smallobjremoved))}")
    print(
        f"Mean tprec (kept):    {Evaluation.evaluation_util.format_score(np.mean(tprecs_inclsmallobj))}")
    print(
        f"Mean tsens (removed): {Evaluation.evaluation_util.format_score(np.mean(tsenss_smallobjremoved))}")
    print(
        f"Mean tsens (kept):    {Evaluation.evaluation_util.format_score(np.mean(tsenss_inclsmallobj))}")

## Evaluate SOI (sub-image of interest) patches

In [ ]:
for original_filename, entries in sorted(soi_groups.items()):
    print(f"\n" + "=" * 60 + f"\nEvaluating SOI: {original_filename}")

    entry = sorted(entries, key=lambda e: int(e["patch"]))[0]
    try:
        img_path, gt_path, pred_path, _ = get_nnunet_case_files(entry)
    except FileNotFoundError as exc:
        print(f"  [WARN] {exc}, skipping.")
        continue

    img_tensor = Evaluation.evaluation_util.load_transform_image(
        img_path)               # (3, 1024, 1024)
    gt_mask = Evaluation.evaluation_util.load_transform_image(
        gt_path, is_mask=True)  # (1, 1024, 1024)
    # Resize prediction to match GT (nnUNet may output at a different resolution)
    pred_mask = Evaluation.pipeline_evaluation_shared.load_binary_mask_as_tensor(
        pred_path, size=gt_mask.shape[-2:])                   # (1, 1024, 1024)

    Evaluation.pipeline_evaluation_shared.evaluate_soi_patch(
        pred_mask, img_tensor, gt_mask)